In [3]:
import numpy as np
import time

# Define the input matrix
matrix = np.array([
    ['a', 'b', 'c'],
    ['a', 'b', 'c'],
    ['a', 'b', 'c']
])

    
size = 256
matrix = np.tile(np.arange(1, (size+1)), (size, 1))
print(matrix)
# Create the rotated version



start_time = time.time()

for i in range (100):
    
    rotated_matrix = np.array([np.roll(matrix[0], shift) for shift in range(len(matrix[0]))])

end_time = time.time()
print("Original Matrix:")
print(matrix)

print("\nRotated Matrix:")
print(rotated_matrix)

time = end_time - start_time
print("time " + str(time) + "s")




[[  1   2   3 ... 254 255 256]
 [  1   2   3 ... 254 255 256]
 [  1   2   3 ... 254 255 256]
 ...
 [  1   2   3 ... 254 255 256]
 [  1   2   3 ... 254 255 256]
 [  1   2   3 ... 254 255 256]]
Original Matrix:
[[  1   2   3 ... 254 255 256]
 [  1   2   3 ... 254 255 256]
 [  1   2   3 ... 254 255 256]
 ...
 [  1   2   3 ... 254 255 256]
 [  1   2   3 ... 254 255 256]
 [  1   2   3 ... 254 255 256]]

Rotated Matrix:
[[  1   2   3 ... 254 255 256]
 [256   1   2 ... 253 254 255]
 [255 256   1 ... 252 253 254]
 ...
 [  4   5   6 ...   1   2   3]
 [  3   4   5 ... 256   1   2]
 [  2   3   4 ... 255 256   1]]
time 0.21709179878234863s


In [21]:
import tensorflow as tf


size = 256
npmatrix = np.tile(np.arange(1, (size+1)), (size, 1))
print(npmatrix)

matrix = tf.convert_to_tensor(npmatrix, dtype=tf.int32)
# Create the rotated version

start_time = time.time()
rotated_matrix = tf.stack([tf.roll(matrix[0]) for shift in range(len(matrix[0]))])

end_time = time.time()
print("Original Matrix:")
print(matrix)

print("\nRotated Matrix:")
print(rotated_matrix)

time = end_time - start_time
print("time " + str(time) + "s")




[[  1   2   3 ... 254 255 256]
 [  1   2   3 ... 254 255 256]
 [  1   2   3 ... 254 255 256]
 ...
 [  1   2   3 ... 254 255 256]
 [  1   2   3 ... 254 255 256]
 [  1   2   3 ... 254 255 256]]


TypeError: roll() missing 1 required positional argument: 'axis'

In [4]:
import tensorflow as tf
import time

size = 256

# Create the matrix using TensorFlow's range and tile functions
matrix = tf.tile(tf.reshape(tf.range(1, size + 1, dtype=tf.int32), (1, size)), (size, 1))

print("Original Matrix:")
print(matrix)

# Create the rotated version by rolling each row
start_time = time.time()

for i in range (100):
# Use tf.roll for each row to create the rotated matrix
    rotated_matrix = tf.stack([tf.roll(matrix[i], shift=i, axis=0) for i in range(size)])

end_time = time.time()

# Print the rotated matrix
print("\nRotated Matrix:")
print(rotated_matrix)

# Calculate and print the elapsed time
elapsed_time = end_time - start_time
print(f"\nTime taken: {elapsed_time:.4f} seconds")



Original Matrix:
tf.Tensor(
[[  1   2   3 ... 254 255 256]
 [  1   2   3 ... 254 255 256]
 [  1   2   3 ... 254 255 256]
 ...
 [  1   2   3 ... 254 255 256]
 [  1   2   3 ... 254 255 256]
 [  1   2   3 ... 254 255 256]], shape=(256, 256), dtype=int32)

Rotated Matrix:
tf.Tensor(
[[  1   2   3 ... 254 255 256]
 [256   1   2 ... 253 254 255]
 [255 256   1 ... 252 253 254]
 ...
 [  4   5   6 ...   1   2   3]
 [  3   4   5 ... 256   1   2]
 [  2   3   4 ... 255 256   1]], shape=(256, 256), dtype=int32)

Time taken: 1.7018 seconds


In [10]:
def create_permutation_matrix(size, shift):
    """
    Creates a permutation matrix for a cyclic shift.

    Parameters:
        size (int): Number of elements in the row.
        shift (int): Number of positions to shift (positive = right, negative = left).

    Returns:
        ndarray: Permutation matrix of shape (size, size).
    """
    shift = shift % size  # Ensure the shift is within bounds
    P = np.eye(size)  # Start with an identity matrix
    P = np.roll(P, shift=shift, axis=1)  # Roll the columns for cyclic rotation
    return P
from scipy.linalg import block_diag

def create_block_diagonal_permutation(matrix):
    """
    Creates a block-diagonal permutation matrix for row-specific rotations.

    Parameters:
        matrix (ndarray): Input matrix to rotate.

    Returns:
        ndarray: Block-diagonal permutation matrix for the rotation.
    """
    num_rows, num_cols = matrix.shape
    # Create a permutation matrix for each row with increasing shift
    permutation_matrices = [create_permutation_matrix(num_cols, shift=i) for i in range(num_rows)]
    # Combine the permutation matrices into a block-diagonal matrix
    return block_diag(*permutation_matrices)

def rotate_matrix_increasing(matrix):
    """
    Rotates the rows of a matrix with increasing shifts using matrix multiplication.

    Parameters:
        matrix (ndarray): Input matrix.

    Returns:
        ndarray: Rotated matrix.
    """
    # Create the block-diagonal permutation matrix
    block_perm = create_block_diagonal_permutation(matrix)
    # Flatten the input matrix for matrix multiplication
    flat_matrix = matrix.flatten(order='C')
    # Apply the block-diagonal permutation
    rotated_flat = block_perm @ flat_matrix
    # Reshape back to the original matrix shape
    return rotated_flat.reshape(matrix.shape)


In [12]:
size = 3
matrix = np.tile(np.arange(1, (size+1)), (size, 1))
print(matrix)


# Perform the rotation
rotated_matrix = rotate_matrix_increasing(matrix)
print("Original Matrix:")
print(matrix)
print("\nRotated Matrix:")
print(rotated_matrix)


[[1 2 3]
 [1 2 3]
 [1 2 3]]
Original Matrix:
[[1 2 3]
 [1 2 3]
 [1 2 3]]

Rotated Matrix:
[[1. 2. 3.]
 [2. 3. 1.]
 [3. 1. 2.]]


In [ ]:
import time
import multiprocessing

def compute_iteration(i):
    sum(x * x for x in range(1))  # Simulate CPU-heavy task
    return i

num_workers = multiprocessing.cpu_count()

print(num_workers)
iterations = 1  # Number of tasks

# Test multiprocessing.Pool
start_time = time.time()
with multiprocessing.Pool(processes=num_workers) as pool:
    results1 = pool.map(compute_iteration, range(iterations))
print("multiprocessing.Pool time:", time.time() - start_time)

'''
# Test ProcessPoolExecutor
start_time = time.time()
with ProcessPoolExecutor(max_workers=num_workers) as executor:
    results2 = list(executor.map(compute_iteration, range(iterations)))
print("ProcessPoolExecutor time:", time.time() - start_time)
'''

16
